In [ ]:
import os
import shutil
import random
import yaml
import cv2
from ultralytics import YOLO

# 1. تحديد المسارات داخل مجلد الـ fine tune
base_dir = '/content/drive/MyDrive/Old_Russian/finetune'
source_images = os.path.join(base_dir, 'images')
source_labels = os.path.join(base_dir, 'labels')
output_dir = os.path.join(base_dir, 'dataset')
best_weights_path = os.path.join(base_dir, 'best.pt')

# 2. إنشاء الهيكلية الجديدة للـ dataset
for split in ['train', 'val']:
    os.makedirs(os.path.join(output_dir, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'labels', split), exist_ok=True)

# 3. دالة معالجة وتوضيح الصور باستخدام OpenCV مع طباعة تقرير لكل صورة
def enhance_image(input_path, output_path):
    img = cv2.imread(input_path)
    if img is None:
        return False

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    denoised = cv2.fastNlMeansDenoising(gray, h=10)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(denoised)
    enhanced_rgb = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)

    cv2.imwrite(output_path, enhanced_rgb)
    return True

# 4. تجميع ونقل البيانات ومعالجتها مع عداد تفصيلي
img_extensions = ('.jpg', '.jpeg', '.png', '.bmp')
all_images = [f for f in os.listdir(source_images) if f.lower().endswith(img_extensions)]

valid_pairs = []
for img_name in all_images:
    base_name = os.path.splitext(img_name)[0]
    label_name = base_name + '.txt'
    lbl_path = os.path.join(source_labels, label_name)

    if os.path.exists(lbl_path) and os.path.getsize(lbl_path) > 0:
        valid_pairs.append((img_name, label_name))

random.seed(42)
random.shuffle(valid_pairs)

num_train = int(len(valid_pairs) * 0.85)
train_pairs, val_pairs = valid_pairs[:num_train], valid_pairs[num_train:]
total_images = len(valid_pairs)

print(f"📊 إجمالي الصور المتطابقة مع الليبلز: {total_images} صورة.")
print(f"🔹 سيتم معالجة وتدريب {len(train_pairs)} صورة، والتحقق من {len(val_pairs)} صورة.\n")

def process_and_copy_with_progress(pairs, split):
    success_count = 0
    for idx, (img_name, lbl_name) in enumerate(pairs, 1):
        src_img = os.path.join(source_images, img_name)
        dst_img = os.path.join(output_dir, 'images', split, img_name)

        success = enhance_image(src_img, dst_img)
        if success:
            shutil.copy(os.path.join(source_labels, lbl_name), os.path.join(output_dir, 'labels', split, lbl_name))
            success_count += 1

        # طباعة حالة التقدم لكل 10 صور أو في النهاية لتشاهد كم عَبَر وكم باقي
        if idx % 10 == 0 or idx == len(pairs):
            print(f"[{split.upper()}] معالجة الصورة رقم {idx} من {len(pairs)} (تم بنجاح: {success_count})")

print("⏳ بدء معالجة وتوضيح صور التدريب...")
process_and_copy_with_progress(train_pairs, 'train')

print("\n⏳ بدء معالجة وتوضيح صور التحقق...")
process_and_copy_with_progress(val_pairs, 'val')

# 5. استخراج الفئات التلقائي وإنشاء ملف data.yaml
max_class_id = 0
for _, lbl_name in valid_pairs:
    lbl_file_path = os.path.join(source_labels, lbl_name)
    with open(lbl_file_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                class_id = int(parts[0])
                if class_id > max_class_id:
                    max_class_id = class_id

num_classes = max_class_id + 1
names_dict = {i: f"class_{i}" for i in range(num_classes)}

yaml_data = {
    'path': output_dir,
    'train': 'images/train',
    'val': 'images/val',
    'names': names_dict
}

yaml_path = os.path.join(output_dir, 'data.yaml')
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(yaml_data, f, default_flow_style=False, allow_unicode=True)

print(f"\n✅ تمت المعالجة وتوليد ملف data.yaml لـ {num_classes} فئة بنجاح.\n")

# 6. تحميل النموذج وبدء الـ Fine-tuning
print("🚀 بدء تدريب النموذج (Fine-tuning) عبر YOLO...")
model = YOLO(best_weights_path)

results = model.train(
    data=yaml_path,
    epochs=120,
    imgsz=1040,
    batch=4,
    lr0=0.0001,
    freeze=10,
    mosaic=0.1,
    box=7.5,
    patience=50,
    project=base_dir,
    name='finetune_enhanced_results'
)

In [4]:
import json
import os
import traceback
import numpy as np
import gradio as gr
from pathlib import Path
from PIL import Image, ImageDraw
from ultralytics import YOLO

# مسار نموذج الـ Fine-Tuning وملف الـ alphabet.json
FINAL_MODEL = Path("/content/drive/MyDrive/Old_Russian/finetune/finetune_enhanced_results/weights/best.pt")
ALPHABET_FILE = Path("/content/drive/MyDrive/Old_Russian/finetune/Alphabet.json")

def load_inference_model():
    model_path = str(FINAL_MODEL)
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"لم يتم العثور على ملف النموذج في المسار: {model_path}")
    return YOLO(model_path)

# تحميل قاموس الحروف بدقة متناهية من ملف alphabet.json المعكوس بناءً على char_to_id
ID_TO_CHAR = {}
if ALPHABET_FILE.exists():
    try:
        with open(ALPHABET_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            # عكس القاموس لربط الـ ID (القيمة) بالحرف الصحيح (المفتاح)
            char_to_id = data.get("char_to_id", {})
            ID_TO_CHAR = {v: k for k, v in char_to_id.items()}
        print(f"✅ تم تحميل وترتيب الحروف بنجاح من ملف alphabet.json! (عدد الحروف: {len(ID_TO_CHAR)})")
    except Exception as e:
        print(f"⚠️ حدث خطأ أثناء قراءة ملف alphabet.json: {e}")

# في حال لم يتم تحميل الملف، يتم الاعتماد على أسماء النموذج الافتراضية كاحتياط
if not ID_TO_CHAR:
    try:
        temp_model = load_inference_model()
        ID_TO_CHAR = temp_model.names
        print("⚠️ تم الاعتماد على أسماء النموذج الداخلية.")
    except:
        pass


def group_boxes_into_text(boxes_xyxy, cls_ids, line_thresh_ratio=0.6, space_factor=0.5):
    if len(boxes_xyxy) == 0:
        return ""

    heights = boxes_xyxy[:, 3] - boxes_xyxy[:, 1]
    median_h = np.median(heights) if len(heights) else 20
    y_centers = (boxes_xyxy[:, 1] + boxes_xyxy[:, 3]) / 2
    order = np.argsort(y_centers)

    lines = []
    current = [order[0]]
    current_y = y_centers[order[0]]

    for idx in order[1:]:
        if abs(y_centers[idx] - current_y) <= median_h * line_thresh_ratio:
            current.append(idx)
        else:
            lines.append(current)
            current = [idx]
            current_y = y_centers[idx]
    lines.append(current)

    output = []
    for line in lines:
        line = sorted(line, key=lambda i: boxes_xyxy[i][0])
        line_chars = []
        widths = [boxes_xyxy[i][2] - boxes_xyxy[i][0] for i in line]
        avg_w = np.mean(widths) if widths else 15

        for i, idx in enumerate(line):
            if i > 0:
                prev_x2 = boxes_xyxy[line[i-1]][2]
                curr_x1 = boxes_xyxy[idx][0]
                gap = curr_x1 - prev_x2
                if gap > (avg_w * space_factor):
                    line_chars.append(" ")

            char = ID_TO_CHAR.get(int(cls_ids[idx]), f"?[{int(cls_ids[idx])}]")
            line_chars.append(char)

        output.append("".join(line_chars))

    return "\n".join(output)


def draw_bboxes(image, boxes_xyxy, cls_ids, confs):
    image = image.convert("RGB").copy()
    draw = ImageDraw.Draw(image)
    for box, cid, conf in zip(boxes_xyxy, cls_ids, confs):
        x1, y1, x2, y2 = map(int, box)
        draw.rectangle([x1, y1, x2, y2], outline="red", width=2)
        char_name = ID_TO_CHAR.get(int(cid), f"?({cid})")
        label = f"{char_name} ({conf:.2f})"
        draw.text((x1, max(0, y1 - 15)), label, fill="blue")
    return image


_INFERENCE_MODEL = None


def process_image(image, conf_thresh, line_ratio):
    global _INFERENCE_MODEL
    try:
        if image is None:
            return None, "الرجاء رفع صورة أولاً."

        if _INFERENCE_MODEL is None:
            _INFERENCE_MODEL = load_inference_model()

        results = _INFERENCE_MODEL.predict(source=image, conf=conf_thresh, imgsz=1056, verbose=False)

        if len(results) == 0 or results[0].boxes is None or len(results[0].boxes) == 0:
            return image, "لم يتم العثور على أي حروف مطابقة بالثقة المحددة."

        boxes = results[0].boxes.xyxy.cpu().numpy()
        cls_ids = results[0].boxes.cls.cpu().numpy()
        confs = results[0].boxes.conf.cpu().numpy()

        annotated = draw_bboxes(image, boxes, cls_ids, confs)
        text = group_boxes_into_text(boxes, cls_ids, line_thresh_ratio=line_ratio)

        return annotated, text

    except Exception:
        err = traceback.format_exc()
        print(err)
        return image, err


# تصميم واجهة Gradio الكاملة
with gr.Blocks(title="OCR — المخطوطات السلافية القديمة") as demo:
    gr.Markdown("# 📜 نظام التعرف البصري على الحروف (OCR) - المخطوطات السلافية القديمة")
    gr.Markdown("قم برفع صورة المخطوط الحقيقي، واضبط إعدادات الكشف، ثم اضغط على **استخراج النص**.")

    with gr.Row():
        with gr.Column(scale=1):
            img_in = gr.Image(type="pil", label="ارفع صورة المخطوط الحقيقي")

            with gr.Accordion("إعدادات متقدمة", open=False):
                conf_slider = gr.Slider(minimum=0.05, maximum=0.9, value=0.15, step=0.05, label="عتبة الثقة (Confidence Threshold)")
                line_slider = gr.Slider(minimum=0.2, maximum=1.5, value=0.6, step=0.1, label="حساسية تجميع الأسطر (Line Spacing Ratio)")

            btn = gr.Button("🔍 استخراج النص وتحديد الحروف", variant="primary")

        with gr.Column(scale=1):
            img_out = gr.Image(label="الصورة مع مربعات التحديد (Bounding Boxes)")
            txt_out = gr.Textbox(label="النص المستخرج", lines=10, max_lines=20)

    btn.click(
        fn=process_image,
        inputs=[img_in, conf_slider, line_slider],
        outputs=[img_out, txt_out]
    )

demo.queue()
demo.launch(share=True, debug=True)

✅ تم تحميل قاموس الحروف بنجاح من ملف classes.txt! (عدد الحروف: 38)
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8d323679bc11fd2b5a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://1a817f7a2b1b239e3a.gradio.live
Killing tunnel 127.0.0.1:7860 <> https://8d323679bc11fd2b5a.gradio.live
